In [1]:
# ===== 0. 載入套件 =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.proportion import proportions_ztest
from IPython.display import display

# 圖表風格設定
sns.set_theme(style="whitegrid")

# ===== 1. 讀取資料 =====
file_path = "../data/raw/YRBS_2007.csv"
df = pd.read_csv(file_path)

# 顯示前幾列，確認資料有成功讀進來
print("資料前5列：")
display(df.head())

# 顯示欄位名稱，確認分析的變數存在
print("\n所有欄位名稱：")
print(df.columns.tolist())

資料前5列：


,RaceEth,HowOldAreYou,WhatIsYourSex,InWhatGradeAreYou,AreYouHispanicOrLatino,WhatIsYourRace,HowTallAreYouWithoutShoesInMeters,HowMuchDoYouWeighWithoutShoesInKG,BicyleHelmetUse,SeatBeltUse,...,InjuredWhileExercising,HIVTesting,SunscreenUse,SunProtection,Sleep,HealthInGeneral,BMIPCT,weight,stratum,psu
0,7.0,4.0,2.0,2.0,1.0,C,NaN,NaN,2.0,1.0,...,3.0,2.0,1.0,1.0,5.0,3.0,NaN,1.5104,101,11030
1,5.0,7.0,2.0,2.0,2.0,E,1.70,68.04,4.0,4.0,...,2.0,3.0,1.0,5.0,4.0,3.0,66.531824,1.8559,101,11030
2,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,5.0,3.0,...,2.0,3.0,2.0,1.0,1.0,1.0,NaN,1.8559,101,11030
3,7.0,1.0,1.0,1.0,1.0,A,1.63,79.38,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,98.174319,1.3264,101,11030
4,7.0,1.0,1.0,5.0,1.0,B,NaN,NaN,6.0,5.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.3264,101,11030



所有欄位名稱：
['RaceEth', 'HowOldAreYou', 'WhatIsYourSex', 'InWhatGradeAreYou', 'AreYouHispanicOrLatino', 'WhatIsYourRace', 'HowTallAreYouWithoutShoesInMeters', 'HowMuchDoYouWeighWithoutShoesInKG', 'BicyleHelmetUse', 'SeatBeltUse', 'RidingWithADrinkingDriver', 'DrinkingAndDriving', 'WeaponCarrying', 'GunCarryingPast12Mos', 'WeaponCarryingAtSchool', 'SafetyConcernsAtSchool', 'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty', 'StolenOrDamagedYourProperty', 'PhysicalFighting', 'InjuredIFight', 'PhysicalFightingAtSchool', 'BoyfriendGirlfriendPhysicallyHurt', 'ForcedSexualIntercourse', 'SadOrHopeless', 'ConsideredSuicide', 'MadeASuicidePlan', 'AttemptedSuicide', 'InjuriousSuicide', 'EverCigaretteUse', 'InitiationSmokingWholeCigarette', 'CurrentCigaretteUse', 'SmokedMoreThan10Cigarettes', 'HowObtainedCigarettes', 'SmokeOnSchoolProperty', 'EverSmokedDailyFor30Days', 'EverSmokingCessation', 'CurrentSmokelessTobaccoUse', 'CurrentSmokelessTobaccoOnSchoolProperty', 'CurrentCigarUse', 'EverAlcoholUs

#### 研究問題 1：高中生中，感到 sad or hopeless 的比例是否與 0.30 不同？
#### 研究問題 2：高中生的平均身高是否與 1.70 公尺不同？

In [2]:
# 比例分析研究問題
print("研究問題 1：高中生中，感到 sad or hopeless 的比例是否與 0.30 不同？")

# 平均數分析研究問題
print("研究問題 2：高中生的平均身高是否與 1.70 公尺不同？")

研究問題 1：高中生中，感到 sad or hopeless 的比例是否與 0.30 不同？
研究問題 2：高中生的平均身高是否與 1.70 公尺不同？


### 2A. 比例分析變數：SadOrHopeless
##### success = 1
##### failure = 2
##### 其他值視為無效或缺失，先排除

In [3]:
prop_var = "SadOrHopeless"

# 看原始次數分配
print("\nSadOrHopeless 原始編碼次數表：")
print(df[prop_var].value_counts(dropna=False).sort_index())

# 只保留有效值 1 和 2
df_prop = df[df[prop_var].isin([1, 2])].copy()

# 建立二元變數：success=1, failure=0
df_prop["SadOrHopeless_binary"] = np.where(df_prop[prop_var] == 1, 1, 0)

# 計算有效樣本數
n_prop = len(df_prop)

# 顯示處理後結果
print("\n比例分析有效樣本數 n =", n_prop)
print("\n重編碼後次數表：")
print(df_prop["SadOrHopeless_binary"].value_counts().sort_index())

# 缺失/無效值數量
invalid_prop = df[prop_var].isna().sum() + (~df[prop_var].isin([1, 2]) & df[prop_var].notna()).sum()
print("\nSadOrHopeless 缺失或無效值數量 =", invalid_prop)


SadOrHopeless 原始編碼次數表：
SadOrHopeless
1.0    4153
2.0    9692
NaN     196
Name: count, dtype: int64

比例分析有效樣本數 n = 13845

重編碼後次數表：
SadOrHopeless_binary
0    9692
1    4153
Name: count, dtype: int64

SadOrHopeless 缺失或無效值數量 = 196


In [4]:
# ============================================
# 2B. 平均數分析變數：HowTallAreYouWithoutShoesInMeters
# 只保留非缺失且合理的身高值
# ============================================

mean_var = "HowTallAreYouWithoutShoesInMeters"

# 先轉成數值，非數字會變成 NaN
df[mean_var] = pd.to_numeric(df[mean_var], errors="coerce")

# 查看原始資料摘要
print("\n身高原始資料摘要：")
print(df[mean_var].describe())

# 先刪除缺失值
df_mean = df[[mean_var]].dropna().copy()

# 只保留合理身高範圍
df_mean = df_mean[(df_mean[mean_var] >= 1.0) & (df_mean[mean_var] <= 2.5)].copy()

# 計算樣本數
n_mean = len(df_mean)
print("\n平均數分析有效樣本數 n =", n_mean)

# 缺失或無效值數量
invalid_mean = df[mean_var].isna().sum() + ((df[mean_var] < 1.0) | (df[mean_var] > 2.5)).sum()
print("身高缺失或無效值數量 =", invalid_mean)
df_mean.to_csv("yrbs_cleaned.csv", index=False)


身高原始資料摘要：
count    13062.000000
mean         1.694038
std          0.101466
min          1.270000
25%          1.630000
50%          1.680000
75%          1.780000
max          2.110000
Name: HowTallAreYouWithoutShoesInMeters, dtype: float64

平均數分析有效樣本數 n = 13062
身高缺失或無效值數量 = 979
